# Stats tests

The numbers are tiny so we should check if they could be noise. We run a paired Wilcoxon test and a bootstrap CI for each signal.

In [ ]:
!pip install scipy pandas numpy

In [ ]:
import os, numpy as np, pandas as pd
from scipy.stats import wilcoxon
np.random.seed(42)
os.makedirs("results", exist_ok=True)

In [ ]:
# upload fairness_comparison.csv
from google.colab import files
files.upload()

In [ ]:
df = pd.read_csv("fairness_comparison.csv")
df.head()

In [ ]:
def boot(x, n=2000):
    x = np.array(x)
    means = [np.random.choice(x, size=len(x), replace=True).mean() for _ in range(n)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

In [ ]:
rows = []
for sig, g in df.groupby("changed_signal"):
    a = g["original_score"].values
    b = g["changed_score"].values
    try:
        stat, p = wilcoxon(a, b)
    except ValueError:
        stat, p = float("nan"), float("nan")
    lo, hi = boot(g["absolute_difference"].values)
    rows.append({
        "changed_signal": sig,
        "n_pairs": len(g),
        "mean_absolute_difference": float(g["absolute_difference"].mean()),
        "bootstrap_ci_low": lo,
        "bootstrap_ci_high": hi,
        "wilcoxon_statistic": float(stat),
        "wilcoxon_p_value": float(p),
    })
out = pd.DataFrame(rows)
out

Pronoun has a tiny p value, so the direction of the shift is consistent. Name and university have larger sizes but the signs cancel.

In [ ]:
out.to_csv("results/fairness_statistical_tests.csv", index=False)
files.download("results/fairness_statistical_tests.csv")